# SRC-001 Network Dataset Audit

## Purpose

Inspect the structure, schema, and basic quality characteristics of the
consolidated network dataset contained in SRC-001.

## Out of scope

This notebook does not:

- preprocess or transform the dataset;
- train machine-learning models;
- analyze PCAP files;
- analyze lower-layer telemetry;
- evaluate RCA performance.

## 1. Setup

Prepare the execution environment and locate the consolidated network dataset
without modifying the raw source.

In [1]:
from src.data.inventory import DataInventory

import pandas as pd

In [2]:
SOURCE_ID = "SRC-001"

inventory = DataInventory(SOURCE_ID)

PROJECT_ROOT = inventory.project_root
RAW_DATA_PATH = inventory.raw_data_path
FILE_EXTENSIONS = inventory.discover_file_extensions()

In [3]:
print("Project root:", PROJECT_ROOT)
print("Raw data path:", RAW_DATA_PATH)
print("File extensions:", FILE_EXTENSIONS)

Project root: /Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform
Raw data path: /Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform/data/raw/SRC-001_netslab_5g_oran_idd
File extensions: {'.csv', '.pcap', '.md', '.txt', '.db'}


In [4]:
NETWORK_EXTENSION = ".csv"

network_files = sorted(
    path
    for path in RAW_DATA_PATH.iterdir()
    if path.is_file() and path.suffix.lower() == NETWORK_EXTENSION
)

network_files

[PosixPath('/Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform/data/raw/SRC-001_netslab_5g_oran_idd/Network_Dataset.csv')]

In [5]:
if not network_files:
    raise FileNotFoundError(
        f"No {NETWORK_EXTENSION} file found in: {RAW_DATA_PATH}"
    )

if len(network_files) > 1:
    raise RuntimeError(
        f"Expected one {NETWORK_EXTENSION} file, found {len(network_files)}."
    )

NETWORK_DATA_PATH = network_files[0]

In [6]:
print("Network dataset:", NETWORK_DATA_PATH.name)
print(f"File size: {NETWORK_DATA_PATH.stat().st_size / (1024 ** 2):.2f} MB")

Network dataset: Network_Dataset.csv
File size: 216.66 MB


## 2. Initial Structure Inspection

Inspect the consolidated network dataset at a structural level before
performing detailed schema and quality analysis.

In [7]:
sample = pd.read_csv(NETWORK_DATA_PATH, nrows=10)

sample

,uid,src_ip,src_port,dst_ip,dst_port,proto,service,duration,src_bytes,dst_bytes,...,dst_ip_bytes,ip_proto,http_trans_depth,attack_category,attack_type,files_total_bytes,is_GET_mthd,http_status_error,is_file_transfered,traffic_type
0,CgvPTm2yEMy1O7Jl9k,192.168.0.1,18728,192.168.0.10,80,tcp,-,0.000041,0.0,0.0,...,40,6,0,ddos,tcp_ack,0.0,0,0,0,1
1,CAOgXB1yv9VtvISnDc,172.16.0.3,29039,192.168.0.10,80,tcp,-,0.000746,0.0,0.0,...,80,6,0,ddos,tcp_ack,0.0,0,0,0,1
2,CXyf8S2WNJVYWlDI15,172.31.0.134,7552,192.168.0.10,80,tcp,-,0.000086,0.0,0.0,...,88,6,0,ddos,syn,0.0,0,0,0,1
3,CXjeD72LNi8Cn2EXZ5,172.16.0.4,59569,52.207.202.187,443,tcp,-,594.651675,73426.0,14841.0,...,55360,6,0,ddos,icmp,0.0,0,0,0,1
4,CgGkxuR9gLRepT5j2,172.16.0.3,42502,192.168.0.10,80,tcp,-,0.000290,0.0,0.0,...,80,6,0,ddos,tcp_ack,0.0,0,0,0,1
5,CsaLg43Xa92E6iLg5a,172.31.0.134,38193,192.168.0.10,123,tcp,-,0.000070,0.0,0.0,...,80,6,0,ddos,syn,0.0,0,0,0,1
6,CF7pg02wVqREfbBmR4,192.168.0.1,64438,192.168.0.10,80,tcp,-,0.000061,0.0,0.0,...,44,6,0,ddos,syn,0.0,0,0,0,1
7,CqGjpJtgHtuaYShhg,192.168.0.1,65476,192.168.0.10,80,tcp,-,1.085667,0.0,254744237.0,...,80,6,0,ddos,tcp_ack,0.0,0,0,0,1
8,CkafH046oxkwZNv928,172.16.0.3,41641,43.245.49.105,3478,udp,-,0.185959,80.0,64.0,...,120,17,0,ddos,slowloris,0.0,0,0,0,1
9,CbBz8O1LV5bOSwCJH9,172.16.0.3,47526,192.168.0.10,80,tcp,-,0.000236,0.0,0.0,...,80,6,0,ddos,tcp_ack,0.0,0,0,0,1


In [8]:
sample.columns.tolist()

['uid',
 'src_ip',
 'src_port',
 'dst_ip',
 'dst_port',
 'proto',
 'service',
 'duration',
 'src_bytes',
 'dst_bytes',
 'conn_state',
 'missed_bytes',
 'history',
 'src_pkts',
 'src_ip_bytes',
 'dst_pkts',
 'dst_ip_bytes',
 'ip_proto',
 'http_trans_depth',
 'attack_category',
 'attack_type',
 'files_total_bytes',
 'is_GET_mthd',
 'http_status_error',
 'is_file_transfered',
 'traffic_type']

In [9]:
sample.head(5).T

,0,1,2,3,4
uid,CgvPTm2yEMy1O7Jl9k,CAOgXB1yv9VtvISnDc,CXyf8S2WNJVYWlDI15,CXjeD72LNi8Cn2EXZ5,CgGkxuR9gLRepT5j2
src_ip,192.168.0.1,172.16.0.3,172.31.0.134,172.16.0.4,172.16.0.3
src_port,18728,29039,7552,59569,42502
dst_ip,192.168.0.10,192.168.0.10,192.168.0.10,52.207.202.187,192.168.0.10
dst_port,80,80,80,443,80
proto,tcp,tcp,tcp,tcp,tcp
service,-,-,-,-,-
duration,0.000041,0.000746,0.000086,594.651675,0.00029
src_bytes,0.0,0.0,0.0,73426.0,0.0
dst_bytes,0.0,0.0,0.0,14841.0,0.0


In [10]:
with NETWORK_DATA_PATH.open("r", encoding="utf-8", errors="replace") as file:
    row_count = sum(1 for _ in file) - 1

column_count = len(sample.columns)

print("Rows:", row_count)
print("Columns:", column_count)

Rows: 1723817
Columns: 26


In [11]:
print(f"Dataset shape: ({row_count}, {column_count})")

Dataset shape: (1723817, 26)


## 3. Schema Analysis

Inspect the dataset schema, column names, and inferred data types without
loading the complete dataset into memory.

In [12]:
columns = sample.columns.tolist()

print(f"Total columns: {len(columns)}")

for index, column in enumerate(columns, start=1):
    print(f"{index:02d}. {column}")

Total columns: 26
01. uid
02. src_ip
03. src_port
04. dst_ip
05. dst_port
06. proto
07. service
08. duration
09. src_bytes
10. dst_bytes
11. conn_state
12. missed_bytes
13. history
14. src_pkts
15. src_ip_bytes
16. dst_pkts
17. dst_ip_bytes
18. ip_proto
19. http_trans_depth
20. attack_category
21. attack_type
22. files_total_bytes
23. is_GET_mthd
24. http_status_error
25. is_file_transfered
26. traffic_type


In [13]:
schema_sample = pd.read_csv(NETWORK_DATA_PATH, nrows=10_000)

In [14]:
dtype_summary = pd.DataFrame(
    {
        "column": schema_sample.columns,
        "dtype": schema_sample.dtypes.astype(str).values,
    }
)

dtype_summary

,column,dtype
0,uid,str
1,src_ip,str
2,src_port,int64
3,dst_ip,str
4,dst_port,int64
5,proto,str
6,service,str
7,duration,float64
8,src_bytes,float64
9,dst_bytes,float64


In [15]:
duplicate_columns = schema_sample.columns[
    schema_sample.columns.duplicated()
].tolist()

unnamed_columns = [
    column
    for column in schema_sample.columns
    if column.lower().startswith("unnamed")
]

whitespace_columns = [
    column
    for column in schema_sample.columns
    if column != column.strip()
]

In [16]:
print("Duplicate columns:", duplicate_columns)
print("Unnamed columns:", unnamed_columns)
print("Columns with surrounding whitespace:", whitespace_columns)

Duplicate columns: []
Unnamed columns: []
Columns with surrounding whitespace: []


## 4. Data Quality

### 4.1 Missing Values

Measure missing values across the complete network dataset using chunked
processing to avoid loading the entire CSV into memory.

In [17]:
CHUNK_SIZE = 100_000

missing_counts = pd.Series(0, index=columns, dtype="int64")
total_rows = 0

for chunk in pd.read_csv(NETWORK_DATA_PATH, chunksize=CHUNK_SIZE):
    missing_counts = missing_counts.add(chunk.isna().sum(), fill_value=0)
    total_rows += len(chunk)

In [18]:
missing_summary = pd.DataFrame(
    {
        "column": missing_counts.index,
        "missing_count": missing_counts.values,
    }
)

missing_summary["missing_pct"] = (
    missing_summary["missing_count"] / total_rows * 100
)

missing_summary = missing_summary.sort_values(
    "missing_count",
    ascending=False
).reset_index(drop=True)

missing_summary

,column,missing_count,missing_pct
0,uid,0,0.0
1,src_ip,0,0.0
2,is_file_transfered,0,0.0
3,http_status_error,0,0.0
4,is_GET_mthd,0,0.0
5,files_total_bytes,0,0.0
6,attack_type,0,0.0
7,attack_category,0,0.0
8,http_trans_depth,0,0.0
9,ip_proto,0,0.0


In [19]:
columns_with_missing = missing_summary[
    missing_summary["missing_count"] > 0
]

columns_with_missing

,column,missing_count,missing_pct


In [20]:
print("Total rows:", total_rows)
print("Columns with missing values:", len(columns_with_missing))

Total rows: 1723817
Columns with missing values: 0


### Finding

No missing values were detected in the network dataset across the complete CSV.

### 4.2 Duplicate Records

Identify exact duplicate rows across the complete network dataset using
chunked processing to avoid loading the entire CSV into memory.

Duplicate records are measured but not removed during this audit.

In [21]:
CHUNK_SIZE = 100_000

seen_hashes = set()
duplicate_count = 0
total_rows = 0

for chunk in pd.read_csv(NETWORK_DATA_PATH, chunksize=CHUNK_SIZE):
    row_hashes = pd.util.hash_pandas_object(chunk, index=False)

    for row_hash in row_hashes:
        if row_hash in seen_hashes:
            duplicate_count += 1
        else:
            seen_hashes.add(row_hash)

    total_rows += len(chunk)

In [22]:
duplicate_pct = duplicate_count / total_rows * 100

print("Total rows:", total_rows)
print("Duplicate rows:", duplicate_count)
print(f"Duplicate percentage: {duplicate_pct:.4f}%")

Total rows: 1723817
Duplicate rows: 83635
Duplicate percentage: 4.8517%


### Finding

The dataset contains 83,635 exact duplicate records, representing approximately
4.85% of the 1,723,817 total rows.

Duplicates are not removed at this stage. Further inspection is required to
determine whether they represent redundant records or legitimate repeated
network observations.

In [23]:
seen_hashes = set()
duplicate_examples = []

for chunk in pd.read_csv(NETWORK_DATA_PATH, chunksize=CHUNK_SIZE):
    row_hashes = pd.util.hash_pandas_object(chunk, index=False)

    duplicate_mask = (
        row_hashes.isin(seen_hashes)
        | row_hashes.duplicated(keep="first")
    )

    if duplicate_mask.any():
        duplicate_examples.append(
            chunk.loc[duplicate_mask]
        )

    seen_hashes.update(row_hashes.tolist())

    if sum(len(df) for df in duplicate_examples) >= 20:
        break

duplicate_examples = pd.concat(
    duplicate_examples,
    ignore_index=True
).head(20)

duplicate_examples

,uid,src_ip,src_port,dst_ip,dst_port,proto,service,duration,src_bytes,dst_bytes,...,dst_ip_bytes,ip_proto,http_trans_depth,attack_category,attack_type,files_total_bytes,is_GET_mthd,http_status_error,is_file_transfered,traffic_type
0,C8eVxf1HXBAZHgOwb8,172.16.0.2,63802,192.168.0.10,80,tcp,http,4.467502,9471.0,43954.0,...,98844,6,1,web,dir_bruteforce,274.0,1,1,1,1
1,C9sKNtBfLOhaQ9Z4h,192.168.0.1,63753,192.168.0.10,80,tcp,http,4.469431,9620.0,43954.0,...,49422,6,1,web,dir_bruteforce,274.0,1,1,1,1
2,CDgPpo24tBy0nqIwo6,192.168.0.1,65146,192.168.0.10,80,tcp,http,3.380429,9457.0,43954.0,...,49526,6,1,web,dir_bruteforce,274.0,1,1,1,1
3,CIvPJx3Uh2ZqjME3Hh,172.16.0.2,63631,192.168.0.10,80,tcp,http,4.969758,9717.0,43954.0,...,98948,6,1,web,dir_bruteforce,274.0,1,1,1,1
4,CNRb0XNToOYaRXC75,172.31.0.134,63341,192.168.0.10,80,tcp,http,6.312388,9650.0,43954.0,...,98844,6,1,web,dir_bruteforce,274.0,1,1,1,1
5,CQFNCm1wMF0uv0OTO5,172.16.0.2,63500,192.168.0.10,80,tcp,http,3.614944,9356.0,43954.0,...,98844,6,1,web,dir_bruteforce,274.0,1,1,1,1
6,CRed6I1fyw9AoKWYZh,172.31.0.134,63881,192.168.0.10,80,tcp,http,6.015029,9792.0,43954.0,...,98844,6,1,web,dir_bruteforce,274.0,1,1,1,1
7,CTY7EN1299CT636yq8,172.16.0.2,63482,192.168.0.10,80,tcp,http,5.669850,9515.0,43954.0,...,98844,6,1,web,dir_bruteforce,274.0,1,1,1,1
8,CTgueM2D005XIX7jJa,172.31.0.134,63659,192.168.0.10,80,tcp,http,3.334767,9675.0,43954.0,...,98844,6,1,web,dir_bruteforce,274.0,1,1,1,1
9,CbYik227UPUSeYVdZc,172.31.0.134,63705,192.168.0.10,80,tcp,http,3.310304,9438.0,43954.0,...,98844,6,1,web,dir_bruteforce,274.0,1,1,1,1


In [24]:
seen_hashes = set()
duplicate_labels = []

for chunk in pd.read_csv(NETWORK_DATA_PATH, chunksize=CHUNK_SIZE):
    row_hashes = pd.util.hash_pandas_object(chunk, index=False)

    duplicate_mask = (
        row_hashes.isin(seen_hashes)
        | row_hashes.duplicated(keep="first")
    )

    duplicate_labels.append(
        chunk.loc[
            duplicate_mask,
            ["attack_category", "attack_type"]
        ]
    )

    seen_hashes.update(row_hashes.tolist())

In [25]:
duplicate_labels = pd.concat(
    duplicate_labels,
    ignore_index=True
)

duplicate_distribution = (
    duplicate_labels
    .groupby(["attack_category", "attack_type"])
    .size()
    .reset_index(name="duplicate_count")
    .sort_values("duplicate_count", ascending=False)
)

duplicate_distribution

,attack_category,attack_type,duplicate_count
0,benign,benign,82819
2,web,sql_injection,458
3,web,xss,341
1,web,dir_bruteforce,17


### Finding

A total of 83,635 exact duplicate records were identified.

Duplicate records are highly concentrated in the benign class:

- 82,819 benign records (approximately 99.02% of all duplicates);
- 458 `sql_injection` records;
- 341 `xss` records;
- 17 `dir_bruteforce` records.

This concentration may affect class distribution and represents a potential
evaluation leakage risk if identical records are distributed across development
and evaluation partitions.

No duplicate records are removed during this audit.

### 4.3 Constant or Empty Columns

Identify columns that contain a single value across the complete dataset.

Constant columns provide no variability and may be irrelevant for subsequent
analysis or modeling.

In [26]:
constant_candidates = {}
variable_columns = set()

for chunk in pd.read_csv(NETWORK_DATA_PATH, chunksize=CHUNK_SIZE):
    for column in chunk.columns:
        if column in variable_columns:
            continue

        values = chunk[column].drop_duplicates()

        if len(values) > 1:
            variable_columns.add(column)
            constant_candidates.pop(column, None)
            continue

        value = values.iloc[0]

        if column not in constant_candidates:
            constant_candidates[column] = value
        elif constant_candidates[column] != value:
            variable_columns.add(column)
            constant_candidates.pop(column, None)

In [27]:
constant_columns = pd.DataFrame(
    {
        "column": list(constant_candidates.keys()),
        "constant_value": list(constant_candidates.values()),
    }
)

constant_columns

,column,constant_value


In [28]:
print("Constant columns:", len(constant_columns))
print("Variable columns:", len(variable_columns))

Constant columns: 0
Variable columns: 26


In [29]:
string_columns = schema_sample.select_dtypes(
    include=["object", "str"]
).columns.tolist()

empty_string_candidates = set(string_columns)

for chunk in pd.read_csv(NETWORK_DATA_PATH, chunksize=CHUNK_SIZE):
    for column in list(empty_string_candidates):
        values = (
            chunk[column]
            .astype("string")
            .fillna("")
            .str.strip()
        )

        if values.ne("").any():
            empty_string_candidates.remove(column)

empty_string_columns = sorted(empty_string_candidates)

empty_string_columns

[]

#### Finding

No constant columns were detected across the complete network dataset.

All 26 columns contain variable data, and no structurally empty string columns
were identified.

### 4.4 Invalid or Unexpected Values

Inspect numeric ranges and categorical values to identify potentially invalid,
unexpected, or inconsistent records.

No values are modified during this audit.

In [30]:
numeric_columns = schema_sample.select_dtypes(
    include="number"
).columns.tolist()

string_columns = schema_sample.select_dtypes(
    include=["object", "str"]
).columns.tolist()

In [31]:
numeric_min = {}
numeric_max = {}

for chunk in pd.read_csv(NETWORK_DATA_PATH, chunksize=CHUNK_SIZE):
    for column in numeric_columns:
        chunk_min = chunk[column].min()
        chunk_max = chunk[column].max()

        numeric_min[column] = min(
            numeric_min.get(column, chunk_min),
            chunk_min,
        )

        numeric_max[column] = max(
            numeric_max.get(column, chunk_max),
            chunk_max,
        )

In [32]:
numeric_range_summary = pd.DataFrame(
    {
        "column": numeric_columns,
        "min": [numeric_min[column] for column in numeric_columns],
        "max": [numeric_max[column] for column in numeric_columns],
    }
)

numeric_range_summary

,column,min,max
0,src_port,0.0,6.552400e+04
1,dst_port,0.0,6.553500e+04
2,duration,0.0,2.040515e+03
3,src_bytes,0.0,4.720238e+09
4,dst_bytes,0.0,3.201801e+09
5,missed_bytes,0.0,1.997054e+09
6,src_pkts,0.0,4.570988e+06
7,src_ip_bytes,0.0,2.091018e+09
8,dst_pkts,0.0,2.889954e+06
9,dst_ip_bytes,0.0,2.987293e+09


In [33]:
categorical_values = {}

for column in string_columns:
    categorical_values[column] = set()

for chunk in pd.read_csv(NETWORK_DATA_PATH, chunksize=CHUNK_SIZE):
    for column in string_columns:
        categorical_values[column].update(
            chunk[column]
            .dropna()
            .astype(str)
            .str.strip()
            .unique()
        )

In [34]:
categorical_summary = pd.DataFrame(
    {
        "column": string_columns,
        "unique_values": [
            len(categorical_values[column])
            for column in string_columns
        ],
    }
)

categorical_summary

,column,unique_values
0,uid,1604266
1,src_ip,318
2,dst_ip,1391
3,proto,4
4,service,15
5,conn_state,13
6,history,3796
7,attack_category,6
8,attack_type,15


In [35]:
numeric_columns = schema_sample.select_dtypes(
    include="number"
).columns.tolist()

observed_values = {
    column: set()
    for column in numeric_columns
}

for chunk in pd.read_csv(NETWORK_DATA_PATH, chunksize=CHUNK_SIZE):
    for column in numeric_columns:
        observed_values[column].update(
            chunk[column].dropna().unique()
        )

In [36]:
binary_columns = [
    column
    for column, values in observed_values.items()
    if values and values.issubset({0, 1})
]

binary_columns

['is_GET_mthd', 'http_status_error', 'is_file_transfered', 'traffic_type']

In [37]:
negative_value_columns = numeric_range_summary[
    numeric_range_summary["min"] < 0
]

negative_value_columns

,column,min,max


In [38]:
LOW_CARDINALITY_LIMIT = 20

low_cardinality_columns = categorical_summary.loc[
    categorical_summary["unique_values"] <= LOW_CARDINALITY_LIMIT,
    "column",
].tolist()

low_cardinality_columns

['proto', 'service', 'conn_state', 'attack_category', 'attack_type']

In [39]:
for column in low_cardinality_columns:
    print(f"\n{column}:")
    print(sorted(categorical_values[column]))


proto:
['icmp', 'tcp', 'udp', 'unknown_transport']

service:
['-', 'ayiya', 'dhcp', 'dns', 'dtls', 'ftp', 'gtpv1', 'http', 'krb_tcp', 'ldap_udp', 'ntp', 'quic', 'quic,ssl', 'ssh', 'ssl']

conn_state:
['OTH', 'REJ', 'RSTO', 'RSTOS0', 'RSTR', 'RSTRH', 'S0', 'S1', 'S2', 'S3', 'SF', 'SH', 'SHR']

attack_category:
['benign', 'bruteforce', 'ddos', 'dos', 'probe', 'web']

attack_type:
['benign', 'dir_bruteforce', 'ftp', 'http_flood', 'icmp', 'os_fingerprinting', 'portscan_tcp', 'portscan_udp', 'slowloris', 'sql_injection', 'ssh', 'syn', 'tcp_ack', 'udp', 'xss']


#### Finding

No negative values were detected in the numeric columns.

Four binary-valued columns were identified automatically, each containing
only `0` and `1`.

Further inspection of low-cardinality categorical values is required before
closing this validation step.

## 5. Label and Scenario Structure

Inspect the target labels and scenario-related fields to understand class
composition and coverage within the network dataset.

### 5.1 Ground-Truth Fields

The official NetsLab-5GORAN-IDD documentation defines six traffic/attack
categories: Benign, DoS, DDoS, Probe, Brute Force, and Web.

`Network_Dataset.csv` is documented as the consolidated upper-layer network
dataset containing information from all attack scenarios.

The exact ground-truth fields used in the CSV must therefore be verified
against the dataset descriptor and the observed CSV schema before being treated
as authoritative labels.

In [40]:
columns

['uid',
 'src_ip',
 'src_port',
 'dst_ip',
 'dst_port',
 'proto',
 'service',
 'duration',
 'src_bytes',
 'dst_bytes',
 'conn_state',
 'missed_bytes',
 'history',
 'src_pkts',
 'src_ip_bytes',
 'dst_pkts',
 'dst_ip_bytes',
 'ip_proto',
 'http_trans_depth',
 'attack_category',
 'attack_type',
 'files_total_bytes',
 'is_GET_mthd',
 'http_status_error',
 'is_file_transfered',
 'traffic_type']

In [41]:
label_combinations = set()

for chunk in pd.read_csv(NETWORK_DATA_PATH, chunksize=CHUNK_SIZE):
    combinations = chunk[
        ["traffic_type", "attack_category", "attack_type"]
    ].drop_duplicates()

    label_combinations.update(
        combinations.itertuples(index=False, name=None)
    )

label_structure = pd.DataFrame(
    sorted(label_combinations),
    columns=[
        "traffic_type",
        "attack_category",
        "attack_type",
    ],
)

label_structure

,traffic_type,attack_category,attack_type
0,0,benign,benign
1,1,bruteforce,ftp
2,1,bruteforce,ssh
3,1,ddos,icmp
4,1,ddos,slowloris
5,1,ddos,syn
6,1,ddos,tcp_ack
7,1,ddos,udp
8,1,dos,icmp
9,1,dos,slowloris


#### Finding

The network dataset contains three hierarchical ground-truth fields:

- `traffic_type`: binary traffic label (`0` = benign, `1` = attack);
- `attack_category`: high-level traffic or attack category;
- `attack_type`: specific attack subtype.

The observed hierarchy is internally consistent:

- benign → benign;
- bruteforce → ftp, ssh;
- ddos → icmp, slowloris, syn, tcp_ack, udp;
- dos → icmp, slowloris, syn, tcp_ack, udp;
- probe → os_fingerprinting, portscan_tcp, portscan_udp;
- web → dir_bruteforce, http_flood, sql_injection, xss.

No inconsistent category/type combinations were observed.

### 5.2 Label Distribution

Measure the distribution of each ground-truth field across the complete
network dataset to identify class imbalance and representation differences.

In [42]:
ground_truth_fields = label_structure.columns.tolist()

ground_truth_fields

['traffic_type', 'attack_category', 'attack_type']

In [43]:
label_counts = {
    column: pd.Series(dtype="int64")
    for column in ground_truth_fields
}

for chunk in pd.read_csv(NETWORK_DATA_PATH, chunksize=CHUNK_SIZE):
    for column in ground_truth_fields:
        counts = chunk[column].value_counts()

        label_counts[column] = (
            label_counts[column]
            .add(counts, fill_value=0)
            .astype("int64")
        )

In [44]:
label_distributions = {}

for column, counts in label_counts.items():
    summary = counts.rename("count").reset_index()
    summary.columns = [column, "count"]

    summary["percentage"] = (
        summary["count"] / total_rows * 100
    )

    label_distributions[column] = (
        summary
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )

In [45]:
for column, summary in label_distributions.items():
    print(f"\n{column}")
    display(summary)


traffic_type


,traffic_type,count,percentage
0,1,1552952,90.087985
1,0,170865,9.912015



attack_category


,attack_category,count,percentage
0,dos,632507,36.692236
1,ddos,420282,24.380894
2,web,288304,16.724745
3,probe,183293,10.632973
4,benign,170865,9.912015
5,bruteforce,28566,1.657136



attack_type


,attack_type,count,percentage
0,syn,506451,29.379627
1,http_flood,254314,14.752958
2,udp,227408,13.192120
3,tcp_ack,199304,11.561784
4,benign,170865,9.912015
5,portscan_tcp,100000,5.801080
6,slowloris,94887,5.504471
7,os_fingerprinting,62957,3.652186
8,dir_bruteforce,28566,1.657136
9,icmp,24739,1.435129


#### Finding

The network dataset presents a substantial class imbalance.

Attack traffic represents approximately 90.09% of all records, while benign
traffic represents approximately 9.91%.

At the category level, `dos` and `ddos` dominate the dataset, together
accounting for more than 60% of all records. `bruteforce` is considerably less
represented.

The imbalance is more pronounced at the attack-type level. `syn` is the most
represented subtype, while categories such as `xss` and `sql_injection`
represent only a small fraction of the dataset.

This imbalance must be considered when defining evaluation partitions and
performance metrics.

### 5.3 Category–Type Consistency

Verify that the hierarchical ground-truth labels are internally consistent.

Each specific `attack_type` should map to a single `attack_category`, and
benign traffic should remain consistently labeled across all ground-truth
fields.

In [46]:
type_category_map = {}

for chunk in pd.read_csv(NETWORK_DATA_PATH, chunksize=CHUNK_SIZE):
    grouped = (
        chunk
        .groupby("attack_type")["attack_category"]
        .unique()
    )

    for attack_type, categories in grouped.items():
        type_category_map.setdefault(attack_type, set()).update(categories)

In [47]:
inconsistent_attack_types = {
    attack_type: sorted(categories)
    for attack_type, categories in type_category_map.items()
    if len(categories) > 1
}

inconsistent_attack_types

{'icmp': ['ddos', 'dos'],
 'slowloris': ['ddos', 'dos'],
 'syn': ['ddos', 'dos'],
 'tcp_ack': ['ddos', 'dos'],
 'udp': ['ddos', 'dos']}

In [48]:
traffic_label_combinations = set()

for chunk in pd.read_csv(NETWORK_DATA_PATH, chunksize=CHUNK_SIZE):
    combinations = chunk[
        ["traffic_type", "attack_category", "attack_type"]
    ].drop_duplicates()

    traffic_label_combinations.update(
        combinations.itertuples(index=False, name=None)
    )

traffic_label_combinations = sorted(traffic_label_combinations)

traffic_label_combinations

[(0, 'benign', 'benign'),
 (1, 'bruteforce', 'ftp'),
 (1, 'bruteforce', 'ssh'),
 (1, 'ddos', 'icmp'),
 (1, 'ddos', 'slowloris'),
 (1, 'ddos', 'syn'),
 (1, 'ddos', 'tcp_ack'),
 (1, 'ddos', 'udp'),
 (1, 'dos', 'icmp'),
 (1, 'dos', 'slowloris'),
 (1, 'dos', 'syn'),
 (1, 'dos', 'tcp_ack'),
 (1, 'dos', 'udp'),
 (1, 'probe', 'os_fingerprinting'),
 (1, 'probe', 'portscan_tcp'),
 (1, 'probe', 'portscan_udp'),
 (1, 'web', 'dir_bruteforce'),
 (1, 'web', 'http_flood'),
 (1, 'web', 'sql_injection'),
 (1, 'web', 'xss')]

#### Finding

The ground-truth hierarchy is internally consistent.

Benign traffic is consistently represented as:

`traffic_type = 0` → `attack_category = benign` → `attack_type = benign`.

All attack records are consistently represented with `traffic_type = 1`.

Some attack types, including `icmp`, `syn`, `udp`, `tcp_ack`, and `slowloris`,
appear under both `dos` and `ddos`.

This does not represent a labeling inconsistency. According to the dataset
documentation, DoS and DDoS may use the same attack mechanisms; the distinction
between both categories depends on how the attack is executed, particularly the
number of attacking UEs involved.

Therefore, the same `attack_type` may validly appear under both `dos` and
`ddos` categories.

## 6. Temporal Structure

Inspect whether the consolidated network dataset preserves explicit temporal
information that can support event ordering, temporal coverage analysis, and
cross-source alignment.

### 6.1 Timestamp Fields

Identify whether the consolidated network dataset contains explicit temporal
fields that can represent the absolute time of each observation.

The inspection considers both columns already recognized as datetime values and
non-numeric columns whose contents can be consistently parsed as timestamps.

Fields representing elapsed time, such as `duration`, are not treated as
absolute timestamps.

#### Finding

No explicit timestamp fields were identified in `Network_Dataset.csv`.

The dataset contains `duration`, but this represents elapsed time rather than
the absolute time at which an observation occurred.

Therefore, timestamp validity, temporal coverage, event ordering, and
cross-source temporal alignment cannot be evaluated from this CSV alone.

In [49]:
candidate_columns = schema_sample.select_dtypes(
    exclude="number"
).columns.tolist()

temporal_candidates = []

for column in candidate_columns:
    values = schema_sample[column].dropna()

    if values.empty:
        continue

    parsed = pd.to_datetime(
        values,
        errors="coerce",
        format="mixed",
    )

    if parsed.notna().all():
        temporal_candidates.append(column)

temporal_candidates

[]

## 7. Leakage Risk Columns

Identify ground-truth fields that must be excluded from the input features
depending on the prediction target.

In [54]:
def get_leakage_risk_columns(target: str, ground_truth_fields: list[str]) -> list[str]:
    if target not in ground_truth_fields:
        raise ValueError(f"Unknown ground-truth target: {target}")

    return [
        column
        for column in ground_truth_fields
        if column != target
    ]

In [55]:
leakage_risk = pd.DataFrame(
    {
        "target": ground_truth_fields,
        "exclude_from_features": [
            get_leakage_risk_columns(target, ground_truth_fields)
            for target in ground_truth_fields
        ],
    }
)

leakage_risk

,target,exclude_from_features
0,traffic_type,"[attack_category, attack_type]"
1,attack_category,"[traffic_type, attack_type]"
2,attack_type,"[traffic_type, attack_category]"


### Finding

The dataset contains three hierarchical ground-truth fields:

- `traffic_type`
- `attack_category`
- `attack_type`

When one field is selected as the prediction target, the other ground-truth
fields must be excluded from the predictive features because they directly
encode information about the target hierarchy.

Exact duplicate records identified earlier must also be handled before
development/evaluation splitting to prevent identical observations from
appearing in both partitions.

## 8. Audit Findings

The audit of `Network_Dataset.csv` produced the following findings:

- The dataset contains 1,723,817 records and 26 variable columns.
- No missing values were detected.
- No constant or structurally empty columns were identified.
- 83,635 exact duplicate records were detected, representing approximately
  4.85% of the dataset.
- Duplicate records are strongly concentrated in benign traffic.
- No negative values were detected in the inspected numeric fields.
- Binary-valued fields contain only the expected `0` and `1` values.
- The dataset provides three hierarchical ground-truth fields:
  `traffic_type`, `attack_category`, and `attack_type`.
- The ground-truth hierarchy is internally consistent. Some attack types occur
  under both DoS and DDoS categories, which is consistent with the documented
  dataset design.
- The dataset is strongly imbalanced toward attack traffic, which represents
  approximately 90.09% of all records.
- DoS and DDoS are the most represented attack categories, while some attack
  types such as `xss` and `sql_injection` have substantially lower
  representation.
- No explicit absolute timestamp field was identified. Temporal ordering and
  temporal coverage therefore cannot be determined from this CSV alone.
- Exact duplicates and hierarchical ground-truth fields represent relevant
  leakage risks that must be controlled when evaluation partitions are
  created.

No data was modified or removed during this audit.

## 9. Network Dataset Suitability Decision

`Network_Dataset.csv` is suitable for use in the DAICP MVP as a structured
network-layer evidence source, but it is not sufficient as a standalone source
for Root Cause Analysis.

### Intended Use

The dataset will be used for:

- **Network traffic characterization:** it provides structured information about
  protocols, ports, connection states, traffic volumes, packet counts, and other
  network-flow characteristics.

- **Ground-truth reference:** the hierarchical fields `traffic_type`,
  `attack_category`, and `attack_type` provide validated labels that can be used
  to identify and evaluate known incident scenarios.

- **Scenario coverage analysis:** the dataset allows the project to determine
  which attack categories and subtypes are represented and how strongly each
  scenario is represented.

- **Evaluation support:** the structured labels and network features can support
  controlled development and evaluation scenarios, provided duplicate records
  and class imbalance are handled before partitioning.

### Limitations

The dataset will not be used as the sole source for RCA because:

- it does not contain explicit absolute timestamps;
- it cannot independently support temporal correlation with other evidence;
- it contains exact duplicate records that must be controlled before evaluation;
- its class distribution is substantially imbalanced;
- it represents processed network observations rather than the complete
  operational context of an incident.

### Decision

`Network_Dataset.csv` is accepted as a structured network-evidence and
ground-truth source for SRC-001.

Additional SRC-001 artifacts, including PCAP captures and lower-layer telemetry,
must be audited before determining whether SRC-001 as a whole provides enough
evidence for the DAICP Root Cause Analysis workflow.